# 12.4 多智能体 (Multi-Agent)

多智能体系统通过协作完成单个Agent难以完成的复杂任务。

本节涵盖：
- 消息传递与协作协议
- 角色扮演与分工
- 辩论式多智能体
- 层级式多智能体
- AutoGen式对话编排
- 产业级多智能体系统设计

## 1. 消息传递与协作协议

多智能体系统的核心是**消息传递协议**：

### 消息结构
```json
{"sender": "agent_A", "receiver": "agent_B", "type": "task_assignment",
 "content": {...}, "timestamp": 1234, "reply_to": null}
```

### 通信模式
- **点对点**：Agent之间直接通信
- **广播**：一个Agent向所有Agent发送消息
- **发布-订阅**：Agent订阅感兴趣的消息类型
- **黑板模式**：共享信息空间，Agent读写黑板

### 产业实践
- AutoGen：基于对话的多Agent框架
- CrewAI：基于角色的多Agent协作
- LangGraph：基于图的多Agent工作流

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from typing import List, Dict, Optional
import json

torch.manual_seed(42)

class Message:
    def __init__(self, sender, receiver, msg_type, content_embed, metadata=None):
        self.sender = sender
        self.receiver = receiver
        self.msg_type = msg_type
        self.content_embed = content_embed.detach().clone()
        self.metadata = metadata or {}
        self.timestamp = 0

class MessageBus:
    def __init__(self):
        self.queues = defaultdict(list)
        self.subscribers = defaultdict(list)
        self.history = []
        self.timestamp = 0

    def send(self, message):
        message.timestamp = self.timestamp
        self.timestamp += 1
        if message.receiver == 'broadcast':
            for agent_id in self.subscribers.get(message.msg_type, []):
                self.queues[agent_id].append(message)
        elif message.receiver == 'publish':
            for agent_id in self.subscribers.get(message.msg_type, []):
                self.queues[agent_id].append(message)
        else:
            self.queues[message.receiver].append(message)
        self.history.append(message)

    def receive(self, agent_id):
        messages = self.queues[agent_id]
        self.queues[agent_id] = []
        return messages

    def subscribe(self, agent_id, msg_type):
        self.subscribers[msg_type].append(agent_id)

class CollaborativeAgent(nn.Module):
    def __init__(self, agent_id, d_model=128, n_actions=4):
        super().__init__()
        self.agent_id = agent_id
        self.d_model = d_model
        self.n_actions = n_actions
        self.message_encoder = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.policy = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_actions)
        )
        self.response_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.inbox_state = torch.zeros(1, d_model)

    def process_messages(self, messages):
        if not messages:
            return self.inbox_state
        encoded = [self.message_encoder(m.content_embed) for m in messages]
        combined = torch.stack(encoded).mean(dim=0)
        self.inbox_state = self.inbox_state * 0.5 + combined * 0.5
        return self.inbox_state

    def act(self, state):
        logits = self.policy(state)
        return logits.argmax(dim=-1).item()

    def respond(self, state, received_embed):
        combined = torch.cat([state, received_embed], dim=-1)
        return self.response_net(combined)

bus = MessageBus()
pm = CollaborativeAgent('PM', d_model=128)
eng = CollaborativeAgent('Engineer', d_model=128)
qa = CollaborativeAgent('QA', d_model=128)

bus.subscribe('Engineer', 'task_assignment')
bus.subscribe('QA', 'review_request')

print('=== Multi-Agent Message Passing ===')
task = torch.randn(1, 128)

msg1 = Message('PM', 'Engineer', 'task_assignment', pm.respond(task, task))
bus.send(msg1)
print(f'PM -> Engineer: task_assignment')

eng_msgs = bus.receive('Engineer')
eng_state = eng.process_messages(eng_msgs)
eng_response = eng.respond(eng_state, task)
msg2 = Message('Engineer', 'QA', 'review_request', eng_response)
bus.send(msg2)
print(f'Engineer -> QA: review_request')

qa_msgs = bus.receive('QA')
qa_state = qa.process_messages(qa_msgs)
qa_action = qa.act(qa_state)
print(f'QA action: {qa_action}')

print(f'\nTotal messages: {len(bus.history)}')
print(f'Message types: {[m.msg_type for m in bus.history]}')
print(f'\nKey: Message bus decouples agents — they communicate through structured messages.')
print(f'Publish-subscribe enables flexible, event-driven communication patterns.')

## 2. 角色扮演与分工协作

**角色扮演**：每个Agent扮演特定角色，按角色职责行动。

**典型角色配置**：
| 角色 | 职责 | 输出 |
|------|------|------|
| 产品经理 | 需求分析、优先级排序 | 需求文档 |
| 架构师 | 系统设计、技术选型 | 设计文档 |
| 工程师 | 编码实现 | 代码 |
| 测试工程师 | 质量验证 | 测试报告 |
| 审查员 | 代码审查 | 审查意见 |

**产业实践**：
- MetaGPT：完整的软件团队角色模拟
- ChatDev：聊天驱动的软件开发
- CrewAI：可配置的Agent团队

In [ ]:
class RoleAgent(nn.Module):
    def __init__(self, name, role, d_model=128, n_actions=4):
        super().__init__()
        self.name = name
        self.role = role
        self.d_model = d_model
        self.role_embed = nn.Parameter(torch.randn(1, d_model) * 0.1)
        self.task_encoder = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.policy = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_actions)
        )
        self.output_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.quality_net = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, task_embed):
        role_task = self.task_encoder(torch.cat([task_embed, self.role_embed.expand(task_embed.size(0), -1)], dim=-1))
        action_logits = self.policy(role_task)
        output = self.output_net(role_task)
        quality = self.quality_net(output)
        return action_logits, output, quality

class SoftwareTeam:
    def __init__(self, d_model=128):
        self.d_model = d_model
        self.pm = RoleAgent('PM_Alice', 'product_manager', d_model)
        self.architect = RoleAgent('Arch_Bob', 'architect', d_model)
        self.engineer = RoleAgent('Eng_Carol', 'engineer', d_model)
        self.tester = RoleAgent('QA_Dave', 'tester', d_model)
        self.reviewer = RoleAgent('Rev_Eve', 'reviewer', d_model)
        self.pipeline = [self.pm, self.architect, self.engineer, self.tester, self.reviewer]
        self.artifacts = {}

    def execute(self, task_embed, max_revision_rounds=2):
        current = task_embed
        for agent in self.pipeline:
            action_logits, output, quality = agent(current)
            self.artifacts[agent.role] = {
                'output': output.detach(),
                'quality': quality.item(),
                'action': action_logits.argmax(dim=-1).item()
            }
            current = output

        for round_num in range(max_revision_rounds):
            review_quality = self.artifacts[self.reviewer.role]['quality']
            if review_quality > 0.8:
                break
            _, eng_output, eng_quality = self.engineer(current)
            _, test_output, test_quality = self.tester(eng_output)
            _, rev_output, rev_quality = self.reviewer(test_output)
            self.artifacts[self.engineer.role] = {'output': eng_output.detach(), 'quality': eng_quality.item()}
            self.artifacts[self.reviewer.role] = {'output': rev_output.detach(), 'quality': rev_quality.item()}
            current = rev_output

        return self.artifacts

team = SoftwareTeam(d_model=128)
task = torch.randn(1, 128)

print('=== Role-Based Software Team ===')
artifacts = team.execute(task, max_revision_rounds=2)

for role, artifact in artifacts.items():
    print(f'{role}: quality={artifact["quality"]:.3f}, action={artifact["action"]}')

print(f'\nKey: Each agent has a role embedding that conditions its behavior.')
print(f'The pipeline flows: PM -> Architect -> Engineer -> Tester -> Reviewer.')
print(f'Revision loops allow iterative improvement until quality threshold is met.')

## 3. 辩论式多智能体

**辩论式**：多个Agent对同一问题提出不同观点，通过辩论达成共识或更优解。

**核心机制**：
1. 每个Agent独立生成初始立场
2. 轮流发言，反驳对方论点
3. 立场随辩论动态调整
4. 裁判/投票决定最终结果

**优势**：
- 减少单一Agent的偏见和幻觉
- 多角度分析提高决策质量
- 辩论过程本身提供可解释性

**产业应用**：
- Multi-Agent Debate (Liang et al.)
- Improving Factuality with Debate
- 人类偏好对齐中的宪法AI (Constitutional AI)

In [ ]:
class DebateAgent(nn.Module):
    def __init__(self, agent_id, d_model=128, stance_dim=32):
        super().__init__()
        self.agent_id = agent_id
        self.d_model = d_model
        self.argument_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.stance_net = nn.Sequential(
            nn.Linear(d_model, stance_dim), nn.ReLU(), nn.Linear(stance_dim, 1), nn.Tanh()
        )
        self.counter_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.update_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )

    def argue(self, topic, prev_arguments=None):
        if prev_arguments is not None and len(prev_arguments) > 0:
            counter = self.counter_net(torch.cat([topic, prev_arguments[-1]], dim=-1))
            argument = self.argument_net(topic + 0.3 * counter)
        else:
            argument = self.argument_net(topic)
        stance = self.stance_net(argument)
        return argument, stance

    def update_stance(self, current_stance, opponent_argument):
        combined = torch.cat([current_stance, opponent_argument], dim=-1)
        return self.update_net(combined)

class DebateJudge(nn.Module):
    def __init__(self, d_model=128, n_agents=3):
        super().__init__()
        self.n_agents = n_agents
        self.evaluate_net = nn.Sequential(
            nn.Linear(d_model * n_agents, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )
        self.consensus_net = nn.Sequential(
            nn.Linear(d_model * n_agents, 64), nn.ReLU(), nn.Linear(64, d_model)
        )

    def judge(self, arguments):
        all_args = torch.cat(arguments, dim=-1)
        quality = self.evaluate_net(all_args)
        consensus = self.consensus_net(all_args)
        return quality, consensus

n_debaters = 3
debaters = [DebateAgent(f'debater_{i}', d_model=128) for i in range(n_debaters)]
judge = DebateJudge(d_model=128, n_agents=n_debaters)

topic = torch.randn(1, 128)

print('=== Multi-Agent Debate ===')
all_arguments = [[] for _ in range(n_debaters)]
stances = []

for round_num in range(4):
    round_args = []
    round_stances = []
    for i, debater in enumerate(debaters):
        prev = [all_arguments[j][-1] for j in range(n_debaters) if j != i and len(all_arguments[j]) > 0]
        prev_arg = prev[0] if prev else None
        argument, stance = debater.argue(topic, [prev_arg] if prev_arg is not None else None)
        all_arguments[i].append(argument)
        round_args.append(argument)
        round_stances.append(stance.item())
    stances.append(round_stances)

    quality, consensus = judge.judge(round_args)
    stance_str = ', '.join([f'{s:+.3f}' for s in round_stances])
    print(f'Round {round_num+1}: stances=[{stance_str}], quality={quality.item():.3f}')

final_quality, final_consensus = judge.judge([args[-1] for args in all_arguments])
print(f'\nFinal debate quality: {final_quality.item():.3f}')
print(f'Stance convergence: round1={[f"{s:+.3f}" for s in stances[0]]} -> round4={[f"{s:+.3f}" for s in stances[-1]]}')

print(f'\nKey: Debate agents argue from different perspectives, converging toward consensus.')
print(f'The judge evaluates overall quality and synthesizes a consensus answer.')

## 4. 层级式多智能体

**层级式**：Agent按层级组织，上层Agent负责规划和决策，下层Agent负责执行。

**典型架构**：
```
            [Supervisor]
           /     |      \
     [Manager1] [Manager2] [Manager3]
     /    \       |         |    \
  [W1]  [W2]   [W3]     [W4]  [W5]
```

**优势**：
- 任务分解自然，上层做决策，下层做执行
- 可扩展性好，增加层级可处理更复杂任务
- 容错性好，下层失败可由上层重新分配

**产业应用**：
- LangGraph Supervisor模式
- AutoGen GroupChat with Manager
- CrewAI Process.hierarchical

In [ ]:
class WorkerAgent(nn.Module):
    def __init__(self, worker_id, specialty, d_model=128):
        super().__init__()
        self.worker_id = worker_id
        self.specialty = specialty
        self.specialty_embed = nn.Parameter(torch.randn(1, d_model) * 0.1)
        self.execute_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.report_net = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def execute(self, task_embed):
        combined = torch.cat([task_embed, self.specialty_embed.expand(task_embed.size(0), -1)], dim=-1)
        result = self.execute_net(combined)
        quality = self.report_net(result)
        return result, quality

class ManagerAgent(nn.Module):
    def __init__(self, manager_id, n_workers, d_model=128):
        super().__init__()
        self.manager_id = manager_id
        self.n_workers = n_workers
        self.assignment_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, n_workers), nn.Softmax(dim=-1)
        )
        self.aggregate_net = nn.Sequential(
            nn.Linear(d_model * n_workers, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.quality_gate = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def assign(self, task_embed):
        return self.assignment_net(task_embed)

    def aggregate(self, worker_results):
        combined = torch.cat(worker_results, dim=-1)
        aggregated = self.aggregate_net(combined)
        quality = self.quality_gate(aggregated)
        return aggregated, quality

class SupervisorAgent(nn.Module):
    def __init__(self, n_managers, d_model=128):
        super().__init__()
        self.n_managers = n_managers
        self.routing_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, n_managers), nn.Softmax(dim=-1)
        )
        self.merge_net = nn.Sequential(
            nn.Linear(d_model * n_managers, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.final_quality = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def route(self, task_embed):
        return self.routing_net(task_embed)

    def merge(self, manager_results):
        combined = torch.cat(manager_results, dim=-1)
        merged = self.merge_net(combined)
        quality = self.final_quality(merged)
        return merged, quality

workers = [
    WorkerAgent(0, 'frontend', d_model=128),
    WorkerAgent(1, 'backend', d_model=128),
    WorkerAgent(2, 'database', d_model=128),
]
manager = ManagerAgent(0, n_workers=3, d_model=128)
supervisor = SupervisorAgent(n_managers=1, d_model=128)

task = torch.randn(1, 128)

print('=== Hierarchical Multi-Agent System ===')
print(f'Supervisor routing: {supervisor.route(task)[0].detach().tolist()}')

assignment = manager.assign(task)
print(f'\nManager assignment: {assignment[0].detach().tolist()}')

worker_results = []
for i, worker in enumerate(workers):
    result, quality = worker.execute(task)
    worker_results.append(result)
    print(f'Worker {i} ({worker.specialty}): quality={quality.item():.3f}')

aggregated, agg_quality = manager.aggregate(worker_results)
print(f'\nManager aggregation quality: {agg_quality.item():.3f}')

final_result, final_quality = supervisor.merge([aggregated])
print(f'Supervisor final quality: {final_quality.item():.3f}')

print(f'\nKey: Hierarchy enables task decomposition and quality control at each level.')
print(f'Supervisor routes tasks, managers assign workers, workers execute.')
print(f'Quality gates at each level enable early rejection of poor results.')

## 5. AutoGen式对话编排

**AutoGen** 是微软开源的多Agent对话框架，核心思想：
- Agent之间通过自然语言对话协作
- 支持人类参与（Human-in-the-loop）
- 可自定义对话模式和终止条件

**对话模式**：
- **Two-Agent Chat**：两个Agent交替对话
- **Group Chat**：多个Agent轮流发言
- **Nested Chat**：对话中嵌套子对话
- **Sequential Chat**：按顺序依次对话

**终止条件**：
- 最大轮数
- 检测到特定关键词（如"TERMINATE"）
- 满足收敛条件（如连续两轮无变化）

In [ ]:
class AutoGenAgent(nn.Module):
    def __init__(self, name, system_message, d_model=128):
        super().__init__()
        self.name = name
        self.system_message = system_message
        self.d_model = d_model
        self.context_encoder = nn.GRU(d_model, 64, num_layers=2, batch_first=True)
        self.response_net = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.terminate_net = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, message_history):
        if len(message_history) == 0:
            return torch.randn(1, self.d_model), 0.0
        encoded_history = torch.cat(message_history[-10:], dim=0).unsqueeze(0)
        _, h = self.context_encoder(encoded_history)
        state = h[-1]
        response = self.response_net(state)
        terminate_prob = self.terminate_net(state).item()
        return response, terminate_prob

class GroupChatManager:
    def __init__(self, agents, d_model=128, max_rounds=6):
        self.agents = agents
        self.d_model = d_model
        self.max_rounds = max_rounds
        self.speaker_selector = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, len(agents)), nn.Softmax(dim=-1)
        )
        self.convergence_detector = nn.Sequential(
            nn.Linear(d_model * 2, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def select_speaker(self, last_message):
        probs = self.speaker_selector(last_message)
        return probs.argmax(dim=-1).item()

    def check_convergence(self, msg1, msg2):
        pair = torch.cat([msg1, msg2], dim=-1)
        return self.convergence_detector(pair).item()

    def run_chat(self, initial_message):
        history = [initial_message]
        transcript = []
        for round_num in range(self.max_rounds):
            speaker_idx = self.select_speaker(history[-1])
            agent = self.agents[speaker_idx]
            response, terminate_prob = agent(history)
            history.append(response)
            transcript.append({'round': round_num + 1, 'speaker': agent.name,
                              'terminate_prob': terminate_prob})
            if terminate_prob > 0.85:
                transcript[-1]['terminated'] = True
                break
            if len(history) >= 3:
                convergence = self.check_convergence(history[-1], history[-2])
                if convergence > 0.9:
                    transcript[-1]['converged'] = True
                    break
        return transcript, history

assistant = AutoGenAgent('Assistant', 'You are a helpful AI assistant.', d_model=128)
critic = AutoGenAgent('Critic', 'You review and improve code.', d_model=128)
coder = AutoGenAgent('Coder', 'You write code based on requirements.', d_model=128)

manager = GroupChatManager([assistant, critic, coder], d_model=128, max_rounds=6)
initial = torch.randn(1, 128)

print('=== AutoGen-Style Group Chat ===')
transcript, history = manager.run_chat(initial)
for entry in transcript:
    extra = ''
    if entry.get('terminated'):
        extra = ' [TERMINATED]'
    if entry.get('converged'):
        extra = ' [CONVERGED]'
    print(f"Round {entry['round']}: {entry['speaker']} (terminate={entry['terminate_prob']:.3f}){extra}")

print(f'\nTotal rounds: {len(transcript)}')
print(f'History length: {len(history)}')
print(f'\nKey: Group chat manager selects speakers and detects convergence.')
print(f'Termination can be triggered by agent decision or conversation convergence.')

## 6. 产业级多智能体系统设计

### 系统架构选择

| 架构 | 优势 | 劣势 | 适用场景 |
|------|------|------|----------|
| 角色流水线 | 简单、可预测 | 无反馈循环 | 结构化任务 |
| 辩论式 | 减少偏见、提高质量 | 成本高、慢 | 需要高质量决策 |
| 层级式 | 可扩展、容错 | 延迟高 | 大规模复杂任务 |
| 对话式 | 灵活、人类可参与 | 难以控制流程 | 探索性任务 |

### 工程挑战

1. **成本控制**：N个Agent的token消耗是单Agent的N倍
   - 解决：使用小模型做Worker，大模型做Supervisor

2. **一致性**：多Agent可能产生矛盾输出
   - 解决：共享上下文、仲裁机制、投票

3. **死锁**：Agent互相等待导致无法推进
   - 解决：超时机制、强制推进、Supervisor仲裁

4. **调试**：多Agent交互复杂，难以复现
   - 解决：完整日志、消息追踪、可视化

5. **安全**：恶意Agent可能注入有害内容
   - 解决：输入过滤、输出审查、权限隔离

In [ ]:
class ProductionMultiAgentSystem:
    def __init__(self, d_model=128, max_cost=1000, max_rounds=10):
        self.d_model = d_model
        self.max_cost = max_cost
        self.max_rounds = max_rounds
        self.supervisor = SupervisorAgent(n_managers=2, d_model=d_model)
        self.managers = [
            ManagerAgent(0, n_workers=2, d_model=d_model),
            ManagerAgent(1, n_workers=2, d_model=d_model),
        ]
        self.workers = [
            [WorkerAgent(0, 'research', d_model), WorkerAgent(1, 'analysis', d_model)],
            [WorkerAgent(2, 'writing', d_model), WorkerAgent(3, 'review', d_model)],
        ]
        self.cost_per_token = 0.01
        self.total_cost = 0
        self.audit_log = []
        self.deadlock_timeout = 3

    def execute(self, task_embed):
        routing = self.supervisor.route(task_embed)
        self._log('supervisor', 'route', routing[0].detach().tolist())

        manager_results = []
        for m_idx, manager in enumerate(self.managers):
            if routing[0, m_idx] < 0.2:
                continue
            assignment = manager.assign(task_embed)
            self._log(f'manager_{m_idx}', 'assign', assignment[0].detach().tolist())

            worker_results = []
            for w_idx, worker in enumerate(self.workers[m_idx]):
                if assignment[0, w_idx] < 0.2:
                    continue
                result, quality = worker.execute(task_embed)
                self.total_cost += self.cost_per_token * 10
                self._log(f'worker_{w_idx}', 'execute', {'quality': quality.item()})
                if quality.item() > 0.3:
                    worker_results.append(result)

            if worker_results:
                agg, agg_quality = manager.aggregate(worker_results)
                self.total_cost += self.cost_per_token * 5
                manager_results.append(agg)

            if self.total_cost >= self.max_cost:
                self._log('system', 'budget_exceeded', {'cost': self.total_cost})
                break

        if manager_results:
            final, final_quality = self.supervisor.merge(manager_results)
            self.total_cost += self.cost_per_token * 3
            self._log('supervisor', 'merge', {'quality': final_quality.item()})
            return final, final_quality.item()
        return None, 0.0

    def _log(self, agent, action, data):
        self.audit_log.append({'agent': agent, 'action': action, 'data': data, 'cost': self.total_cost})

    def get_stats(self):
        return {'total_cost': self.total_cost, 'audit_entries': len(self.audit_log)}

system = ProductionMultiAgentSystem(d_model=128, max_cost=1.0)
task = torch.randn(1, 128)

print('=== Production Multi-Agent System ===')
result, quality = system.execute(task)
print(f'Final quality: {quality:.3f}')
print(f'System stats: {system.get_stats()}')

print(f'\nAudit log:')
for entry in system.audit_log:
    print(f'  [{entry["agent"]}] {entry["action"]} (cost={entry["cost"]:.2f})')

print(f'\nKey: Production systems need cost budgets, quality gates, and audit logs.')
print(f'Skipping low-relevance workers saves cost without sacrificing quality.')
print(f'Supervisor routing determines which teams are activated per task.')

## 7. 多智能体总结

| 协作模式 | 通信方式 | 决策机制 | 适用场景 |
|---------|---------|---------|----------|
| 角色流水线 | 顺序传递 | 各角色独立 | 软件开发 |
| 辩论式 | 轮流发言 | 投票/裁判 | 决策、事实核查 |
| 层级式 | 上行下行 | 上层决策 | 大规模任务 |
| 对话式 | 自由对话 | 收敛/终止 | 探索性任务 |
| 黑板式 | 读写共享区 | 竞争写入 | 并行问题求解 |

**设计原则**：
1. **最小必要通信**：减少Agent间不必要的消息传递
2. **质量门控**：每层设置质量阈值，拒绝低质量输出
3. **成本感知**：根据任务重要性分配计算资源
4. **可观测性**：完整日志和追踪，便于调试
5. **容错设计**：超时、重试、降级策略

**前沿方向**：
- **AgentVerse**：可配置的多Agent环境
- **CAMEL**：角色扮演的自主协作
- **RoCo**：机器人协作的多Agent系统
- **Mixture-of-Agents (MoA)**：多Agent集成提升输出质量